# Project 03 — Estimating a Rate (Poisson with exposure offset)

**Scenario.** Genomics counts per unit exposure: each sample $i$ has an exposure $e_i$ (e.g. sequencing depth in kb) and an observed count $y_i$ of events occurring at a common rate $\lambda$ per unit exposure.

**New skill:** priors for a positive rate via a **log link** ($\lambda = e^{\text{log\_rate}}$). **Key pitfall:** ignoring the exposure/offset. Because exposures *vary*, the correct model is $y_i \sim \text{Poisson}(e_i\,\lambda)$ — forgetting $e_i$ biases the rate.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240603

## Step 1 — Problem & data-generating story

Each sample's count is Poisson with mean $e_i\,\lambda$: the rate per unit exposure is shared, but the exposure differs by sample. **Assumptions made explicit:** (a) events independent (Poisson), (b) a single common rate $\lambda$ (no overdispersion / no covariates), (c) exposures $e_i$ known exactly. We synthesize from a known $\text{log\_rate}=-1.2$ ($\lambda\approx 0.30$) with exposures spread over $[5, 40]$ so the offset genuinely matters.

In [ ]:
from data.generate_data import generate
data = generate()
y, e = data['y'], data['exposure']
print(f"n={data['n']}, total events={y.sum()}, total exposure={e.sum():.1f}")
print(f"naive mean count (IGNORES exposure) = {y.sum()/data['n']:.3f}")
print(f"exposure-adjusted rate = {y.sum()/e.sum():.3f}  (true rate = {np.exp(data['truth']['log_rate']):.3f})")

Notice the two summaries already disagree: the *naive* mean count per sample (~7) is nothing like the true rate (~0.30). The difference is entirely the exposure. A plot of count vs exposure shows the tell-tale upward trend: samples with more exposure rack up more events at the *same* underlying rate.

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5))
ax.scatter(e, y, color='#4C72B0')
ax.set(xlabel='exposure e_i', ylabel='count y_i',
       title='counts rise with exposure at a fixed underlying rate')
plt.tight_layout()

## Step 2 — Model specification (likelihood + justified priors)

$$y_i \sim \text{Poisson}(e_i\,\lambda), \qquad \lambda = e^{r}, \qquad r \equiv \text{log\_rate} \sim \text{Normal}(0, 2).$$

**Why a log link?** A rate must be positive. Putting a Normal prior on $\text{log\_rate}$ makes $\lambda=e^{r}$ automatically positive and the prior symmetric on the log (multiplicative) scale — the natural scale for a rate. $\text{Normal}(0,2)$ is weakly-informative: it covers roughly $e^{-4}$ to $e^{4}$, i.e. rates from $\sim 0.02$ to $\sim 55$. **The offset.** The expected count is $e_i\,\lambda$; equivalently $\log E[y_i] = \log e_i + r$, so $\log e_i$ enters as a fixed **offset**.

In [ ]:
from model import build_model, fit
model = build_model(data)   # use_offset=True by default
model

## Step 3 — Prior predictive checks

We simulate counts implied by the prior (with the real exposures). $\text{Normal}(0,2)$ on log_rate should imply a wide but not insane range of total counts. A prior that placed all its mass on huge rates would imply millions of events; one too tight would forbid plausible rates.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=500, random_seed=RNG)
obs_dim = [d for d in prior.prior_predictive['y'].dims if d not in ('chain','draw')][0]
prior_tot = prior.prior_predictive['y'].sum(dim=obs_dim).values.ravel()
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(np.log10(prior_tot + 1), bins=30, color='#55A868', edgecolor='white')
ax.set(xlabel='log10(total events + 1) implied by prior', ylabel='count',
       title='prior predictive — wide but finite')
plt.tight_layout()

## Step 4 — Inference (NUTS)

We sample with `draws=1000, tune=1000, chains=4`. The single parameter log_rate lives on an unconstrained scale (thanks to the log link), so the geometry is benign and NUTS mixes well.

In [ ]:
idata = fit(data, draws=1000, tune=1000, chains=4, seed=303)

## Step 5 — Computational diagnostics

Check $\hat R\approx 1.00$, bulk/tail ESS $\gtrsim 400$, and 0 divergences. The log link gives a smooth, unconstrained posterior, so a clean report is expected.

In [ ]:
print(az.summary(idata, var_names=['log_rate']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))
print('true log_rate =', data['truth']['log_rate'])

In [ ]:
az.plot_trace(idata, var_names=['log_rate']); plt.tight_layout()

## Step 6 — Posterior predictive checks

We compare observed counts to posterior-predictive replicates. Because the model uses the exposure offset, the predicted counts should track the observed counts *across the full exposure range* — not just on average.

In [ ]:
az.plot_ppc(idata, num_pp_samples=100); plt.tight_layout()

In [ ]:
pp = idata.posterior_predictive['y']
obs_dim = [d for d in pp.dims if d not in ('chain','draw')][0]
pp_tot = pp.sum(dim=obs_dim).values.ravel()
p_value = float(np.mean(pp_tot >= y.sum()))
print(f'observed total={y.sum()}; posterior-predictive p-value for total={p_value:.3f} (near 0.5 = good fit)')

## Step 7 — Model criticism & comparison: the offset matters

We criticize the model by fitting the **wrong** version that omits the exposure offset, and comparing. The correct model recovers the truth; the no-offset model is badly biased — it mistakes 'more exposure -> more counts' for 'a higher rate'.

In [ ]:
idata_no_offset = fit(data, draws=1000, tune=1000, chains=2, seed=303,
                      use_offset=False)
print('correct  log_rate mean =', float(idata.posterior['log_rate'].mean()))
print('no-offset log_rate mean =', float(idata_no_offset.posterior['log_rate'].mean()))
print('true     log_rate      =', data['truth']['log_rate'])
print('the no-offset estimate ~ log(mean count) ~ log(7) ~ 1.95, far from -1.2')

## Step 8 — Decision & communication

Report the rate on the interpretable (per-unit-exposure) scale, with a credible interval, and translate to an expected count for a stated exposure.

In [ ]:
r_post = idata.posterior['log_rate'].values.ravel()
rate_post = np.exp(r_post)
lo, hi = np.percentile(rate_post, [3, 97])
print(f'Rate lambda = {rate_post.mean():.3f} events/unit, 94% CI [{lo:.3f}, {hi:.3f}]')
exposure_q = 20.0
exp_count = rate_post.mean() * exposure_q
print(f'Expected count at exposure {exposure_q}: ~{exp_count:.1f} events')

**Conclusion (for a collaborator).** The event rate is about 0.30 per unit exposure (94% CI roughly [0.27, 0.36]). A new sample with exposure 20 should yield about 6 events. The single most important modeling decision was using the **exposure offset**: without it the rate is off by an order of magnitude.